In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import GWFish.modules as gw
import gc
import torch
from dingo.core.posterior_models.build_model import build_model_from_kwargs
import pandas as pd
from dingo.gw.inference.sampler import GWComposedSampler
import dingo.gw.injection as injection
from dingo.gw.noise.asd_dataset import ASDDataset
import shutil



os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # use GPU 1
os.environ["taskset -c"] = "0-48"  # use CPU cores 0-48
waveform_model = 'IMRPhenomXPHM'
f_ref = 20.
detectors = ['ET']
network = gw.detection.Network(detector_ids = detectors, detection_SNR = (0., 8.))


/home/tpausch/.conda/envs/ET-DINGO/lib/python3.10/site-packages/lalsimulation/_lalsimulation_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


## Define functions

The template bank is built from the intrinsic prior. Each injection is filtered with templates projected onto the same detector network and using that injection's extrinsic parameters. The matched-filter phase is maximized analytically; see the comments in the filtering cell for the limitations of this approximation.


In [2]:
def simulate_injections(name, num_injections):

    pm = build_model_from_kwargs(
        filename=f"/scratch/tpausch/models/{name}/model_latest.pt",
        device="cuda",
        load_training_info=False,
    )

    
    out_dir = f"/scratch/tpausch/test_data/{name}/injections"
    os.makedirs(out_dir, exist_ok=True)

    np.random.seed(42)  # optional, for reproducibility

    injection_generator = injection.Injection.from_posterior_model_metadata(
        pm.metadata
    )

    asd_fname = "/scratch/tpausch/ASD/asds_ET_T8.hdf5"
    asd_dataset = ASDDataset(file_name=asd_fname)

    injection_generator.asd = {
        k: v[0] for k, v in asd_dataset.asds.items()
    }

    # This is the combined intrinsic + extrinsic bilby PriorDict
    # that the network was trained on
    prior = injection_generator.prior
    print(prior)

    # ============================================================
    # Draw and run injections
    # ============================================================

    for idx in range(num_injections):

        print(f"Running injection {idx}")

        theta = prior.sample()
        theta = {k: float(v) for k, v in theta.items()}  # cast numpy -> float

        strain_data = injection_generator.injection(theta)

        payload = {
            "theta_true": theta,
            "strain_data": strain_data,
        }

        fname = os.path.join(out_dir, f"inj_{idx:04d}.pkl")

        with open(fname, "wb") as f:
            pickle.dump(payload, f)

    print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
    print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
    del pm
    gc.collect()
    torch.cuda.empty_cache()
    return 1 

In [3]:
# Build a template bank over the intrinsic prior.
#
# The bank stores intrinsic parameter combinations. Detector-projected
# waveforms are generated for each injection using that injection's
# extrinsic parameters (sky position, inclination, distance and coalescence
# time). This avoids accidentally using one fixed ET antenna response for
# every injection.

from itertools import product
from dingo.gw.injection import Injection

MODEL_NAME = "stage1"  # e.g. "50_1000"
MODEL_PATH = f"/scratch/tpausch/models/{MODEL_NAME}/model_latest.pt"

# Read the model metadata once. Injection.from_posterior_model_metadata
# reproduces its waveform generator, detector list, domain and reference time.
pm = build_model_from_kwargs(
    filename=MODEL_PATH,
    device="cpu",
    load_training_info=False,
)
template_generator = Injection.from_posterior_model_metadata(pm.metadata)
template_metadata = pm.metadata
del pm
gc.collect()

# Explicitly use the domain and detector configuration reported by the model.
detectors = template_metadata["train_settings"]["data"]["detectors"]
print("Template detectors:", detectors)
print("Template domain:", template_metadata["dataset_settings"]["domain"])

# Coarse bank resolution. Reduce the grid for a quick test; refine it after
# checking the recovery on a few injections.
CHIRP_MASSES = np.arange(50.0, 1000.0 + 0.1, 5.0)
MASS_RATIOS = np.arange(0.125, 1.0001, 0.125)
#MASS_RATIOS = [1.0]
SPIN_VALUES = [0.0, 0.45, 0.9]
SPIN_PAIRS = list(product(SPIN_VALUES, repeat=2))

def component_masses_from_mc_q(chirp_mass, mass_ratio):
    """Return (m1, m2), with q=m2/m1 <= 1."""
    q = mass_ratio
    m1 = chirp_mass * (1.0 + q)**(1.0/5.0) / q**(3.0/5.0)
    return m1, q * m1

def make_template_bank(chirp_masses=CHIRP_MASSES,
                       mass_ratios=MASS_RATIOS,
                       spin_pairs=SPIN_PAIRS):
    """Grid covering the model's intrinsic prior support."""
    bank = []
    for mc, q, (chi1, chi2) in product(
        chirp_masses, mass_ratios, spin_pairs
    ):
        m1, m2 = component_masses_from_mc_q(mc, q)

        # Apply the model's component-mass constraints as well as its
        # chirp-mass and mass-ratio bounds.
        if not (30.0 <= m2 <= m1 <= 3000.0):
            continue

        bank.append({
            "chirp_mass": float(mc),
            "mass_ratio": float(q),
            "chi_1": float(chi1),
            "chi_2": float(chi2),
        })
    return bank

template_bank = make_template_bank()
print(f"Created {len(template_bank):,} intrinsic templates.")



WARNING!

This object was created using Dingo version 0.1.dev1230+gc9da5d765 < 0.9.0, which broke backwards compatibility.
For models trained prior to this change, new inference results will be unreliable.
Previously-generated result files should be used with caution.
Reasons for backward compatibility breaking:

v0.9.0: Change to window factor usage, see https://git.ligo.org/pe/pe-group-coordination/-/issues/1#note_1469386.



Putting posterior model to device cpu.
Setting spin_conversion_phase = 0.0. Using this value for the phase parameter for conversion to cartesian spins.
Template detectors: ['ET-1', 'ET-2', 'ET-3']
Template domain: {'type': 'UniformFrequencyDomain', 'f_min': 6.0, 'f_max': 256.0, 'delta_f': 0.125}
Created 13,428 intrinsic templates.


In [ ]:
# Matched-filter each saved injection against the intrinsic template bank.
#
# The statistic below is the network matched-filter SNR, maximized over a
# common coalescence phase. The injection's known extrinsic parameters are
# held fixed. This is useful for estimating/recovering chirp mass in a
# simulation study, but it is NOT a blind search over sky position, distance,
# inclination or coalescence time.

from pathlib import Path

INJECTION_DIR = f"/scratch/tpausch/test_data/{MODEL_NAME}/injections"
RESULTS_DIR = f"/scratch/tpausch/test_data/{MODEL_NAME}/matched_filter"
os.makedirs(RESULTS_DIR, exist_ok=True)

F_MIN = 6.0
F_MAX = 256.0
DELTA_F = 0.125

def get_injection_asds(event):
    """Support the Dingo Injection output key used in this notebook."""
    if "asd" in event:
        return event["asd"]
    if "asds" in event:
        return event["asds"]
    raise KeyError(
        f"No ASD found in injection. Available keys: {list(event.keys())}"
    )

def network_inner_products(data, template, asds, delta_f=DELTA_F):
    """
    Return (d|h) and (h|h), summed over detectors.

    Dingo injections are unwhitened frequency-domain strains. We therefore
    use the one-sided PSD S_n(f) = ASD(f)^2 and the standard 4*delta_f sum.
    """
    cross = 0.0j
    template_norm_sq = 0.0

    common_detectors = sorted(set(data) & set(template) & set(asds))
    if not common_detectors:
        raise ValueError(
            "No common detector keys in waveform, template and ASD: "
            f"{list(data)}, {list(template)}, {list(asds)}"
        )

    for det in common_detectors:
        d = np.asarray(data[det]).reshape(-1)
        h = np.asarray(template[det]).reshape(-1)
        asd = np.asarray(asds[det]).reshape(-1)

        n = min(len(d), len(h), len(asd))
        d, h, asd = d[:n], h[:n], asd[:n]
        freqs = np.arange(n) * delta_f

        valid = (
            (freqs >= F_MIN) & (freqs <= F_MAX)
            & np.isfinite(asd) & (asd > 0)
            & np.isfinite(d.real) & np.isfinite(d.imag)
            & np.isfinite(h.real) & np.isfinite(h.imag)
        )
        if not np.any(valid):
            continue

        psd = asd[valid]**2
        cross += 4.0 * delta_f * np.sum(
            np.conjugate(h[valid]) * d[valid] / psd
        )
        template_norm_sq += 4.0 * delta_f * np.sum(
            np.abs(h[valid])**2 / psd
        )

    return cross, float(np.real(template_norm_sq))


def make_template_parameters(theta_true, intrinsic):
    """
    Copy all extrinsic parameters from the injection and replace only the
    four intrinsic template parameters.
    """
    theta = dict(theta_true)

    # The injection prior may also have stored derived component masses.
    # Remove them so the waveform generator derives m1 and m2 from the
    # template's chirp_mass and mass_ratio rather than retaining the
    # injection's true component masses.
    theta.pop("mass_1", None)
    theta.pop("mass_2", None)
    theta.update(intrinsic)

    # Keep the phase fixed at zero: its unknown value is handled by taking
    # abs((d|h)) in the matched-filter statistic.
    theta["phase"] = 0.0
    return theta


def matched_filter_one_injection(payload, bank, generator):
    event = payload["strain_data"]
    theta_true = payload["theta_true"]

    data = event["waveform"]
    asds = get_injection_asds(event)

    best = {
        "snr": -np.inf,
        "chirp_mass": np.nan,
        "mass_ratio": np.nan,
        "chi_1": np.nan,
        "chi_2": np.nan,
    }

    for j, intrinsic in enumerate(bank):
        theta_template = make_template_parameters(theta_true, intrinsic)

        # This uses Dingo's own waveform generator and detector projection:
        # same IMRPhenomXPHM, f_ref, frequency domain and ET detector geometry.
        generated = generator.signal(theta_template)
        template = generated["waveform"]

        cross, norm_sq = network_inner_products(data, template, asds)
        if norm_sq <= 0 or not np.isfinite(norm_sq):
            continue

        # Maximize analytically over a common phase.
        snr = np.abs(cross) / np.sqrt(norm_sq)

        if snr > best["snr"]:
            best.update(intrinsic)
            best["snr"] = float(snr)

        if (j + 1) % 500 == 0:
            print(f"  evaluated {j + 1:,}/{len(bank):,} templates", end="\r")

    return best


def run_matched_filter(injection_dir=INJECTION_DIR,
                       bank=template_bank,
                       generator=template_generator,
                       max_injections=None):
    files = sorted(Path(injection_dir).glob("inj_*.pkl"))
    if max_injections is not None:
        files = files[:max_injections]

    if not files:
        raise FileNotFoundError(f"No inj_*.pkl files found in {injection_dir}")

    rows = []
    for i, fname in enumerate(files):
        print(f"\nFiltering injection {i + 1}/{len(files)}: {fname.name}")
        with open(fname, "rb") as f:
            payload = pickle.load(f)

        result = matched_filter_one_injection(payload, bank, generator)
        theta_true = payload["theta_true"]

        rows.append({
            "file": fname.name,
            "chirp_mass_true": theta_true["chirp_mass"],
            "chirp_mass_mf": result["chirp_mass"],
            "mass_ratio_true": theta_true["mass_ratio"],
            "mass_ratio_mf": result["mass_ratio"],
            "chi_1_true": theta_true.get("chi_1", np.nan),
            "chi_1_mf": result["chi_1"],
            "chi_2_true": theta_true.get("chi_2", np.nan),
            "chi_2_mf": result["chi_2"],
            "network_snr_mf": result["snr"],
        })

    results = pd.DataFrame(rows)
    output_file = os.path.join(RESULTS_DIR, "matched_filter_results.csv")
    results.to_csv(output_file, index=False)
    print(f"\nSaved results to {output_file}")
    return results


# Start with a small number of injections to verify the pipeline.
# Set max_injections=None to process the complete directory.
results_mf = run_matched_filter(max_injections=5)

display(results_mf)

# Recovery diagnostic.
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    results_mf["chirp_mass_true"],
    results_mf["chirp_mass_mf"],
    c=results_mf["network_snr_mf"],
    cmap="viridis",
    s=45,
)
limits = [
    min(results_mf["chirp_mass_true"].min(), results_mf["chirp_mass_mf"].min()),
    max(results_mf["chirp_mass_true"].max(), results_mf["chirp_mass_mf"].max()),
]
ax.plot(limits, limits, "k--", label="Perfect recovery")
ax.set(
    xlabel=r"True chirp mass [$M_\odot$]",
    ylabel=r"Matched-filter chirp mass [$M_\odot$]",
    title="Matched-filter chirp-mass recovery",
)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
relative_error = (
    results_mf["chirp_mass_mf"] - results_mf["chirp_mass_true"]
) / results_mf["chirp_mass_true"]
ax.scatter(results_mf["chirp_mass_true"], relative_error)
ax.axhline(0.0, color="k", linestyle="--")
ax.set(
    xlabel=r"True chirp mass [$M_\odot$]",
    ylabel="Relative chirp-mass error",
)
ax.grid(alpha=0.3)
plt.show()



Filtering injection 1/5: inj_0000.pkl
